In [2]:
#%matplotlib inline

import os
import subprocess
import itertools
import numpy as np
import requests
import pytz
import datetime
import netCDF4
from osgeo import gdal
from os import path
from osgeo.gdalconst import *
from tqdm import tqdm
from bs4 import BeautifulSoup


In [3]:
url_catalog = 'https://opendap.deltares.nl/thredds/catalog/opendap/rijkswaterstaat/jarkus/grids/catalog.html'
url_base = 'http://opendap.deltares.nl/thredds/dodsC/opendap/rijkswaterstaat/jarkus/grids'
ext = 'nc'
urls = []


def listFD(url, ext=''):
    page = requests.get(url).text
    soup = BeautifulSoup(page, 'html.parser')

    return [url + '/' + node.get('href') for node in soup.find_all('a') if node.get('href').endswith(ext)]


for ncfile in listFD(url_catalog, ext):
    items = ncfile.split('/catalog.html/')
    filename = items[1].split('/')[-1]
    url = url_base + '/' + filename
    if filename == 'catalog.nc':
        continue
    urls.append(url)

In [4]:
urls[:]


['http://opendap.deltares.nl/thredds/dodsC/opendap/rijkswaterstaat/jarkus/grids/jarkusKB111_4948.nc',
 'http://opendap.deltares.nl/thredds/dodsC/opendap/rijkswaterstaat/jarkus/grids/jarkusKB111_5150.nc',
 'http://opendap.deltares.nl/thredds/dodsC/opendap/rijkswaterstaat/jarkus/grids/jarkusKB112_4746.nc',
 'http://opendap.deltares.nl/thredds/dodsC/opendap/rijkswaterstaat/jarkus/grids/jarkusKB112_4948.nc',
 'http://opendap.deltares.nl/thredds/dodsC/opendap/rijkswaterstaat/jarkus/grids/jarkusKB113_4544.nc',
 'http://opendap.deltares.nl/thredds/dodsC/opendap/rijkswaterstaat/jarkus/grids/jarkusKB113_4746.nc',
 'http://opendap.deltares.nl/thredds/dodsC/opendap/rijkswaterstaat/jarkus/grids/jarkusKB113_4948.nc',
 'http://opendap.deltares.nl/thredds/dodsC/opendap/rijkswaterstaat/jarkus/grids/jarkusKB114_4342.nc',
 'http://opendap.deltares.nl/thredds/dodsC/opendap/rijkswaterstaat/jarkus/grids/jarkusKB114_4544.nc',
 'http://opendap.deltares.nl/thredds/dodsC/opendap/rijkswaterstaat/jarkus/grids/ja

In [5]:
grids = []
for url in tqdm(urls[:]):
    ds = netCDF4.Dataset(url)
    times = netCDF4.num2date(ds.variables['time'][:], ds.variables['time'].units, calendar='julian')
    local = pytz.timezone("Europe/Amsterdam")
    # times = [local.localize(t, is_dst=None).astimezone(pytz.utc) for t in times]
    times = [datetime.datetime.strptime(t.isoformat(), "%Y-%m-%dT%H:%M:%S").replace(tzinfo=pytz.utc) for t in times]
    arrs = []
    z = ds.variables['z'][:]
    x = ds.variables['x'][:]
    y = ds.variables['y'][:]

    grids.append({
        "url": url,
        "x": x,
        "y": y,
        "z": z,
        "times": times
    })
    ds.close()


100%|██████████| 63/63 [18:07<00:00, 17.26s/it]


In [6]:
count = len(list(itertools.chain.from_iterable([g['times'] for g in grids])))
count

3094

In [7]:
print(grids[0]['z'][0])

[[-- -- -- ... -- -- --]
 [-- -- -- ... -- -- --]
 [-- -- -- ... -- -- --]
 ...
 [-- -- -- ... -- -- --]
 [-- -- -- ... -- -- --]
 [-- -- -- ... -- -- --]]


In [8]:

#cmd
#subprocess.call('gsutil cp '../output/bathymetry_1985_0001.tif' gs://eo-bathymetry-rws/vaklodingen/bathymetry_1985_0001.tif', shell=True)
#ccc=r"dir"
#ccc
#subprocess.call(ccc)

In [9]:
# Make sure you create the image collection folder in google earth engine before running

In [10]:
ee_collection_path = 'projects/deltares-rws/eo-bathymetry/jarkus'

In [11]:
def run(cmd, shell=True):
    # print(cmd)
    subprocess.call(cmd,shell=shell)

In [12]:
#for g in tqdm(grids):
#    print(g['times'])

In [13]:
start_index = 0
dirbathy = r'../output_jarkusgrids/'
j = 0
ts = []
if not os.path.exists(dirbathy):
    os.makedirs(dirbathy)
for g in tqdm(grids):
    ncols = len(g['x'])
    nrows = len(g['y'])
    cellsize = g['x'][1] - g['x'][0]
    # taking corners
    xllcorner = np.min(g['x']-10)
    yllcorner = np.min(g['y']-10)
    nodata_value = -32767
    z = g['z']
    #print(z.shape)

    for i, t in enumerate(g['times']):
        ts.append(t)
        if i < start_index:
            i = i + 1
            continue
        j += 1
        filename = 'jarkusgrids_' + str(str(t)[:4]) + '_' + str(j).rjust(4, '0')
        filepath = dirbathy  + filename
        filepath_asc = filepath + '.asc'
        filepath_tif = filepath + '.tif'

        zi = z[i]

        with open(filepath_asc, 'w') as f:
            f.write('ncols {0}\n'.format(ncols))
            f.write('nrows {0}\n'.format(nrows))
            f.write('cellsize {0}\n'.format(cellsize))
            f.write('xllcorner {0}\n'.format(xllcorner))
            f.write('yllcorner {0}\n'.format(yllcorner))
            f.write('nodata_value {0}\n'.format(nodata_value))
            for row in range(nrows-1,-1,-1):
                s = ' '.join([str(v) for v in zi[row,]]).replace('--', str(nodata_value))
                f.write(s)
                f.write('\n')

        #cmd = 'gdal_translate -ot Float32 -a_srs EPSG:28992 -co COMPRESS=DEFLATE -co PREDICTOR=2 -co ZLEVEL=6 -of GTiff {0} {1}'\
        #    .format(filepath_asc, filepath_tif)
        # per tile
        cmd = 'gdal_translate -ot Float32 -a_srs EPSG:28992 -of COG {0} {1}'\
            .format(filepath_asc, filepath_tif)
        run(cmd)


  0%|          | 0/63 [00:00<?, ?it/s]

Input file size is 500, 625
0...10...20...30...40...50...60...70...80...90...100 - done.
Input file size is 500, 625
0...10...20...30...40...50...60...70...80...90...100 - done.
Input file size is 500, 625
0...10...20...30...40...50...60...70...80...90...100 - done.
Input file size is 500, 625
0...10...20...30...40...50...60...

  2%|▏         | 1/63 [00:03<03:06,  3.01s/it]

70...80...90...100 - done.
Input file size is 500, 625
0...10...20...30...40...50...60...70...80...90...100 - done.
Input file size is 500, 625
0...10...20...30...40...50...60...70...80...90...100 - done.
Input file size is 500, 625
0...10...20...30...40...50...60...70...80...90...100 - done.
Input file size is 500, 625
0...10...20...30...40...50...60...

  3%|▎         | 2/63 [00:06<03:11,  3.14s/it]

...

.20...30...40...50...60...70...80...90...100 - done.
Input file size is 500, 625
0...10...20...30...40...50...60...70...80...90...100 - done.
Input file size is 500, 625
0...10...20...30...40...50...60...70...80...90...100 - done.
Input file size is 500, 625
0...10...20...30...40...50...60...70...80...90...100 - done.
Input file size is 500, 625
0...10...20...30...40...50...60...70...80...90...100 - done.
Input file size is 500, 625
0...10...20...30...40...50...60...70...80...90...100 - done.
Input file size is 500, 625
0...10...20...30...40...50...60...70...80...90...100 - done.
Input file size is 500, 625
0...10...20...30...40...50...60...70...80...90...100 - done.
Input file size is 500, 625
0...10...20...30...40...50...60...70...80...90...100 - done.
Input file size is 500, 625
0...10...20...30...40...50...60...70...80...90...100 - done.
Input file size is 500, 625
0...10...20...30...40...50...60...70...80...90...100 - done.
Input file size is 500, 625
0...10...20...30...40...50...

 98%|█████████▊| 62/63 [42:37<00:31, 31.48s/it]

.40...50...60...70...80...90...100 - done.
Input file size is 500, 625
0...10...20...30...40...50...60...70...80...90...100 - done.
Input file size is 500, 625
0...

100%|██████████| 63/63 [42:39<00:00, 40.62s/it]

10...20...30...40...50...60...70...80...90...100 - done.


In [14]:
nodata_value = -32767

In [17]:
# import ee
# ee.Authenticate()

Enter verification code:  4/1Ab32j90xcw2SQLszshvM5xIybNfQ37o6_C1P7BUpOfMaDhfBN7inLNVd190



Successfully saved authorization token.


In [ ]:
# merge per year
tzinfo = ts[0].tzinfo
uyears = list(dict.fromkeys(map(lambda x: x.year, ts))) # unique years
uts = list(map(lambda x: datetime.datetime(year=x, month=1, day=1).replace(tzinfo=tzinfo), uyears)) # unique times

for ii, tt in tqdm(enumerate(uyears)):
    filename = 'jarkusgrids_' + str(str(tt)[:4])
    filepath = dirbathy + filename
    filepath_tif = [dirbathy+ll for ll in os.listdir(dirbathy) if str(tt) in ll.split('_')[1] and ll.endswith('.tif')]
    filepath_year_tif = filepath + '.tif'
    
    # per year
    files_to_mosaic = filepath_tif 
    g = gdal.Warp(filepath_year_tif, files_to_mosaic, dstSRS='EPSG:28992', 
                  outputType=gdal.GDT_Float32, format="COG",
                      creationOptions=["COMPRESS=LZW"])
    g = None 
    
    filepath_gs = 'gs://eo-bathymetry-rws/jarkusgrids/' + filename  # temporary file system in storage bucket
    #print(filepath_gs)
    cmd = 'gsutil cp {0} {1}' \
        .format(filepath_year_tif, filepath_gs)
    run(cmd, shell=True)

    filepath_ee = ee_collection_path + '/' + filename
    #print(filepath_ee)
    cmd = 'earthengine upload image --wait --asset_id={0} --nodata_value={1} {2}' \
        .format(filepath_ee, nodata_value, filepath_gs)
    run(cmd, shell=True)

    time_start = int(uts[ii].timestamp() * 1000)
    cmd = 'earthengine asset set --time_start {0} {1}' \
        .format(time_start, filepath_ee)
    run(cmd, shell=True)

    cmd = 'earthengine acl set public {0}' \
        .format(filepath_ee)
    run(cmd, shell=True)


0it [00:00, ?it/s]Copying file://../output_jarkusgrids/jarkusgrids_2015.tif [Content-Type=image/tiff]...
| [1 files][ 12.8 MiB/ 12.8 MiB]                                                
Operation completed over 1 objects/12.8 MiB.                                     


Started upload task with ID: GMIV53AD5TZVZFW6SPDLWM3P
Waiting for the upload task to complete...
Task GMIV53AD5TZVZFW6SPDLWM3P ended at state: FAILED after 10.25 seconds
Error: Cannot overwrite asset 'projects/earthengine-legacy/assets/projects/deltares-rws/eo-bathymetry/jarkus/jarkusgrids_2015'.


1it [00:34, 34.97s/it]

Parent of the asset 'projects/deltares-rws/eo-bathymetry/jarkus/jarkusgrids_2015' is a collection; operation not allowed.


Copying file://../output_jarkusgrids/jarkusgrids_2016.tif [Content-Type=image/tiff]...
| [1 files][ 12.4 MiB/ 12.4 MiB]                                                
Operation completed over 1 objects/12.4 MiB.                                     


...

34it [24:58, 41.38s/it]

Parent of the asset 'projects/deltares-rws/eo-bathymetry/jarkus/jarkusgrids_1997' is a collection; operation not allowed.


Copying file://../output_jarkusgrids/jarkusgrids_1998.tif [Content-Type=image/tiff]...
\ [1 files][ 10.4 MiB/ 10.4 MiB]                                                
Operation completed over 1 objects/10.4 MiB.                                     


Started upload task with ID: NIIAFLKPAA66BLZRB3QGP4NX
Waiting for the upload task to complete...
Task NIIAFLKPAA66BLZRB3QGP4NX ended at state: FAILED after 7.93 seconds
Error: Cannot overwrite asset 'projects/earthengine-legacy/assets/projects/deltares-rws/eo-bathymetry/jarkus/jarkusgrids_1998'.


Traceback (most recent call last):
  File "/opt/conda/envs/geo-env/lib/python3.14/site-packages/urllib3/connection.py", line 198, in _new_conn
    sock = connection.create_connection(
        (self._dns_host, self.port),
    ...<2 lines>...
        socket_options=self.socket_options,
    )
  File "/opt/conda/envs/geo-env/lib/python3.14/site-packages/urllib3/util/connection.py", line 60, in create_connection
    for res in socket.getaddrinfo(host, port, family, socket.SOCK_STREAM):
               ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/envs/geo-env/lib/python3.14/socket.py", line 983, in getaddrinfo
    for res in _socket.getaddrinfo(host, port, family, type, proto, flags):
               ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
socket.gaierror: [Errno -3] Temporary failure in name resolution

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/opt/conda/envs/geo-env/li

In [1]:
# following is just for testing.

In [ ]:
        filepath_gs = 'gs://eo-bathymetry-rws/jarkus/' + filename_tif
        
        #gsutil = 'D:/src/google-cloud-sdk/bin/gsutil.cmd' # relative path is not defined on Windows
        gsutil = 'gsutil'
        cmd = gsutil + ' cp {0} {1}'\
            .format(filepath_tif, filepath_gs)
        run(cmd)
        
        filepath_ee = ee_collection_path + '/' + filename        
        cmd = 'earthengine upload image --wait --asset_id={0} --nodata_value={1} {2}'\
            .format(filepath_ee, nodata_value, filepath_gs)        
        run(cmd)
        
        time_start = int(grids[0]['times'][0].timestamp() * 1000)
        cmd = 'earthengine asset set --time_start {0} {1}'\
            .format(time_start, filepath_ee)
        run(cmd)

        cmd = 'earthengine acl set public {0}'\
            .format(filepath_ee)
        run(cmd)
